# Hierarchical Indices

Two-tier retrieval: find the right section via summaries, then search chunks only inside it.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *
import numpy as np

sections = split_sections(load_text())
summaries = [llm(f"Summarise in one sentence:\n\n{body}") for _, body in sections]
summary_store = build_store([f"{h}: {s}" for (h, _), s in zip(sections, summaries)])

chunks, meta = [], []
for i, (heading, body) in enumerate(sections):
    for c in chunk_text(body, size=300, overlap=30):
        chunks.append(c); meta.append({"section": i})
chunk_store = build_store(chunks, meta)
print(len(sections), "summaries,", len(chunks), "chunks")

In [ ]:
def hierarchical_search(query, top_sections=2, k=3):
    wanted = {h["idx"] for h in summary_store.search(query, top_sections)}
    mask = np.array([m["section"] in wanted for m in meta])
    return chunk_store.search(query, k, mask=mask), wanted

query = "How does Microsoft GraphRAG answer global questions?"
hits, wanted = hierarchical_search(query)
print("Sections searched:", [sections[i][0] for i in wanted])
show(hits, width=100)
print(answer(query, [h["text"] for h in hits]))

Searching 2 small candidate sets is faster and less noisy than searching every chunk, especially on large corpora.